# InfiniDopamine Continued Pretraining (CPT) — Multimodal

This notebook trains **InfiniDopamine** with **Continued Pretraining (CPT)** using Unsloth-accelerated PyTorch/TRL on **streaming** Hugging Face datasets.

## What this notebook does

1. Builds an **InfiniDopamine multimodal model** (`InfiniDopamineForConditionalGeneration`) with the **Qwen3.5 vision tower** inherited and loads weights from **Qwen/Qwen3.5-0.8B**.
2. Accepts a **list of datasets** (default: 17 datasets) via `streaming=True` + `interleave_datasets` so nothing is fully loaded into RAM.
3. **Formats each dataset** according to its actual schema (`messages`, `text`, `prompt`/`response`, `steps`, `Moves`, base64 frames, etc.).
4. Injects a **per-token reward signal** (NLL/CE/perplexity) into the model's existing `reward_values` forward API.
5. Trains with **LoRA** (or full fine-tune) using a custom `SFTTrainer.compute_loss`.
6. Saves checkpoints, can **merge LoRA** for deployment, and optionally pushes to the Hub.

> **Note:** `InfiniDopamineForConditionalGeneration` supports `reward_values` and gate regularization loss, matching the text-only `InfiniDopamineForCausalLM`.

In [ ]:
%pip install --quiet \
  "unsloth>=2024.7" \
  "transformers>=5.0.0" \
  "torch>=2.9.0" \
  "trl>=0.12.0" \
  "peft>=0.7.0" \
  "datasets>=2.20.0" \
  "accelerate>=0.34.0" \
  "bitsandbytes>=0.43.0" \
  "sentencepiece" \
  "tokenizers" \
  "einops" \
  "tensorboard" \
  "git+https://github.com/Gabz4200/QwenDopamine.git" \
  "Pillow" \
  "numpy" \
  "tqdm"


In [ ]:
import gc
import os
import datetime
import json
import base64
import io
from typing import Any, Dict, List, Optional, Union
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from datasets import interleave_datasets, load_dataset, IterableDataset
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoProcessor,
    AutoTokenizer,
    TrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from trl import SFTTrainer

# Register InfiniDopamine so HF Auto classes can instantiate it.
from qwendopamine.integrations.huggingface import HFIntegration
from qwendopamine.models.infinidopamine import (
    InfiniDopamineConfig,
    InfiniDopamineForCausalLM,
    InfiniDopamineForConditionalGeneration,
    InfiniDopamineTextConfig,
    InfiniDopamineVisionConfig,
)

HFIntegration.register_infinidopamine_hf()

# Unsloth / Ampere+ optimization.
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

print("InfiniDopamine registered.")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device         : {torch.cuda.get_device_name(0)}")

## Dataset notes

### Tier 1 — must-use fully
- **SMB** (`DylanRiden/smb-worldmodel-data`): Repo-only dataset with `smb_frames.zip`; download + extract `.npz` frames, serialize 8-button action vectors to text.
- **WorldModelForMaze** (`Kalso42/WorldModelForMaze`): Code repo with maze text files under `data/maze/`; clone and read `.txt` maze layouts as text.
- **Sokoban** (`ultrastar111/...`): `messages` column is a JSON string of trajectory turns; `task`, `env_id`, `init_state`, `stats` also available.
- **Bytesized32** (`thuml/bytesized32-world-model-cot`): `prompt` list + `reward_model` JSON + `extra_info` JSON.
- **PatronusAI** (`PatronusAI/world_model_corpus`): `messages` list of `{role, content}` dicts. Config is `train`, split is `train`.
- **ARC** (`schema-harness/arc-agi-3-schema-traces`): **#1 most important dataset.** Benchmark results with `task` ID, `status`, `win_levels`, and `level0`–`level9` numeric scores. No raw puzzle grids in this split; serialize performance metadata to text.
- **Chess (laion)** (`laion/strategic_game_chess`): `Moves` list of SAN strings + `Termination` + `Result` → PGN-like text.
- **OpenThoughts** (`ryanmarten/OpenThoughts-1k-sample`): `system` + `conversations` list of `{from, value}`.
- **ALFWorld** (`Decix/ReBel-ALFWorld-SFT-Trajectories`): `steps` JSON string → serialized observation/action transcript.

### Tier 2 — important, use a subset for Kaggle speed
- **Kimi K3 coding** (`greghavens/...`): `messages` list with `{role, content, reasoning_content}`.
- **CoT eval** (`cot-leaderboard/cot-eval-traces-2.0`): `passage` + `question` + `options` + `answer` + `reasoning_trace` (often empty). Split is `test`.
- **Lichess** (`Lichess/standard-chess-games`): PGN columns (`Event`, `White`, `Black`, `ECO`, `Opening`, `movetext`, etc.).
- **ToolACE** (`lockon/ToolACE`): `system` + `conversations` list of `{from, value}`.
- **Qwen3.8 distillation** (`faunix/...`): `messages` list with `{role, content}`, plus `domain`/`category` metadata.
- **Fable-5** (`Glint-Research/Fable-5-traces`): Config `pi_agent`; `messages` list + `trace` string + `prompt`.
- **WikiText** (`Salesforce/wikitext`): Single `text` column; use `wikitext-103-raw-v1` config; some empty rows.

### Tier 3 — high-value distillation
- **r0b0tlab** (`r0b0tlab/...`): `messages_json` JSON string → parse + flatten. Configs: `sft_balanced`, `canonical`, `glm47_native`.

In [ ]:
# ──────────────────────────────────────────────────────────────
#  CPT Configuration
# ──────────────────────────────────────────────────────────────

BASE_MODEL_NAME: str = "Qwen/Qwen3.5-0.8B"

# Accepts one or many datasets. Each entry is streamed independently
# and interleaved, so nothing is fully loaded into RAM.
CPT_DATASETS: List[str] = [
    # ── Tier 1: must-use fully ──────────────────────────────
    "DylanRiden/smb-worldmodel-data",        # .npz frames → action text
    "Kalso42/WorldModelForMaze",             # maze .txt files
    "ultrastar111/sokoban_easy_v8_cot_chunk_kinf_world_model_20260707_perseg",
    "thuml/bytesized32-world-model-cot",
    "PatronusAI/world_model_corpus",
    "schema-harness/arc-agi-3-schema-traces",  # ← #1 most important
    "laion/strategic_game_chess",
    "ryanmarten/OpenThoughts-1k-sample",
    "Decix/ReBel-ALFWorld-SFT-Trajectories",
    # ── Tier 2: important, can subset ───────────────────────
    "greghavens/kimi-k3-coding-and-debugging-traces",
    "cot-leaderboard/cot-eval-traces-2.0",
    "Lichess/standard-chess-games",
    "lockon/ToolACE",
    "faunix/Qwen3.8-27B-Distillation-40K",
    "Glint-Research/Fable-5-traces",
    "Salesforce/wikitext",
    # ── Tier 3: high-value distillation ─────────────────────
    "r0b0tlab/qwen3.8-max-glm5.2-kimi-k3-distillation",
]

# Column used as raw text for standard streaming datasets.
# Special datasets (SMB, maze, ALFWorld, chess, etc.) override this.
DATASET_TEXT_COLUMN: str = "text"

# Context window for CPT.
MAX_SEQ_LENGTH: int = 2048

# Precision
TORCH_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
LOAD_IN_4BIT: bool = False

# LoRA / PEFT
USE_LORA: bool = True
LORA_R: int = 16
LORA_ALPHA: int = 32
LORA_DROPOUT: float = 0.05
LORA_TARGET_MODULES = [
    # Standard LLM projections
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
    # InfiniDopamine GDN-2 projections
    "in_proj_qkv", "in_proj_z", "in_proj_a", "in_proj_b", "in_proj_w",
    "in_proj_gate",
    # Reward-net output projection
    "output_proj",
    # Vision tower (Qwen3-VL style)
    "merger.linear_fc1",
    "merger.linear_fc2",
]

# Reward signal for the GatedRewardNet / GatedDeltaNet blocks.
# nll = negative log-likelihood  |  ce = cross-entropy  |  ppl = perplexity-like
REWARD_LOSS_TYPE: str = "nll"  # "nll" | "ce" | "ppl"
REWARD_SCALE: float = 1.0
REWARD_EVERY_N_STEPS: int = 1   # 1 = every step; >1 saves compute

# Training hyperparameters
PER_DEVICE_TRAIN_BATCH_SIZE: int = 2
GRADIENT_ACCUMULATION_STEPS: int = 8
LEARNING_RATE: float = 1e-4      # ~10% of pretrain LR for CPT
WEIGHT_DECAY: float = 0.01
LR_SCHEDULER_TYPE: str = "cosine"
WARMUP_STEPS: int = 100
NUM_TRAIN_EPOCHS: int = 1
LOGGING_STEPS: int = 10
SAVE_STEPS: int = 500
SAVE_TOTAL_LIMIT: int = 2

# Hub / checkpointing
OUTPUT_DIR: str = f"./infini-dopamine-cpt-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"
RESUME_FROM_CHECKPOINT: Optional[str] = None
HUB_MODEL_ID: str = os.environ.get("HUB_MODEL_ID", "")
PUSH_TO_HUB: bool = bool(HUB_MODEL_ID)
HF_TOKEN: Optional[str] = os.environ.get("HF_TOKEN")
MERGE_LORA_AFTER_TRAINING: bool = True

# SMB dataset cache
SMB_CACHE_DIR: str = "./smb-cache"

# Maze dataset cache
MAZE_CACHE_DIR: str = "./maze-cache"

# Subset controls for huge datasets (Kaggle speed)
MAX_ROWS_PER_DATASET: Optional[int] = None  # None = use all
LICHESS_MAX_ROWS: int = 500_000
COT_EVAL_MAX_ROWS: int = 100_000
R0B0TLAB_MAX_ROWS: int = 200_000
WIKITEXT_MAX_ROWS: int = 50_000

## Build InfiniDopamine & load Qwen3.5 weights

We construct an `InfiniDopamineConfig` that mirrors Qwen3.5-0.8B (including the vision tower), instantiate `InfiniDopamineForConditionalGeneration`, and then copy overlapping weights from the base model.

Any weight that exists only in InfiniDopamine (GDN-2 projections, reward-net betas/gates, sliding-window params, etc.) stays at its randomly-initialized value.

In [ ]:
def _as_obj(d: dict) -> Any:
    class _Cfg:
        pass
    c = _Cfg()
    for k, v in d.items():
        setattr(c, k, v)
    return c


def build_text_config_from_qwen(qwen_cfg: Any) -> InfiniDopamineTextConfig:
    """Build an InfiniDopamineTextConfig that mirrors the Qwen3.5-0.8B text backbone."""
    src = getattr(qwen_cfg, "text_config", qwen_cfg)
    if isinstance(src, dict):
        src = _as_obj(src)

    cfg_kwargs = dict(
        vocab_size=getattr(src, "vocab_size", 248320),
        hidden_size=getattr(src, "hidden_size", 1024),
        intermediate_size=getattr(src, "intermediate_size", 3584),
        num_hidden_layers=getattr(src, "num_hidden_layers", 24),
        num_key_value_heads=getattr(src, "num_key_value_heads", 2),
        max_position_embeddings=getattr(src, "max_position_embeddings", 262144),
        rms_norm_eps=getattr(src, "rms_norm_eps", 1e-6),
        rope_theta=getattr(src, "rope_theta", 10_000_000.0),
        attention_bias=getattr(src, "attention_bias", False),
        hidden_act=getattr(src, "hidden_act", "silu"),
        sliding_window=getattr(src, "sliding_window", None),
        attn_output_gate=getattr(src, "attn_output_gate", True),
        full_attention_interval=getattr(src, "full_attention_interval", 4),
        linear_conv_kernel_dim=getattr(src, "linear_conv_kernel_dim", 4),
        linear_key_head_dim=getattr(src, "linear_key_head_dim", 128),
        linear_num_key_heads=getattr(src, "linear_num_key_heads", 16),
        linear_num_value_heads=getattr(src, "linear_num_value_heads", 16),
        linear_value_head_dim=getattr(src, "linear_value_head_dim", 128),
        mlp_only_layers=getattr(src, "mlp_only_layers", []),
        mamba_ssm_dtype=getattr(src, "mamba_ssm_dtype", "float32"),
    )
    layer_types = getattr(src, "layer_types", None)
    if layer_types is not None:
        cfg_kwargs["layer_types"] = layer_types

    rope_parameters = getattr(src, "rope_parameters", None)
    if rope_parameters is not None:
        cfg_kwargs["rope_parameters"] = rope_parameters

    return InfiniDopamineTextConfig(**cfg_kwargs)


def build_vision_config_from_qwen(qwen_cfg: Any) -> InfiniDopamineVisionConfig:
    """Build an InfiniDopamineVisionConfig from Qwen3.5 vision config."""
    src = getattr(qwen_cfg, "vision_config", qwen_cfg)
    if isinstance(src, dict):
        src = _as_obj(src)

    cfg_kwargs = dict(
        depth=getattr(src, "depth", 12),
        hidden_size=getattr(src, "hidden_size", 768),
        in_channels=getattr(src, "in_channels", 3),
        intermediate_size=getattr(src, "intermediate_size", 3072),
        model_type=getattr(src, "model_type", "qwen3_5"),
        num_heads=getattr(src, "num_heads", 12),
        num_position_embeddings=getattr(src, "num_position_embeddings", 2304),
        out_hidden_size=getattr(src, "out_hidden_size", 1024),
        patch_size=getattr(src, "patch_size", 16),
        spatial_merge_size=getattr(src, "spatial_merge_size", 2),
        temporal_patch_size=getattr(src, "temporal_patch_size", 2),
        hidden_act=getattr(src, "hidden_act", "gelu_pytorch_tanh"),
        initializer_range=getattr(src, "initializer_range", 0.02),
    )
    return InfiniDopamineVisionConfig(**cfg_kwargs)


tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME, trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Qwen3.5 uses a Qwen3-VL-style processor for images/video.
try:
    processor = AutoProcessor.from_pretrained(
        BASE_MODEL_NAME, trust_remote_code=True
    )
except Exception as e:
    print(f"[WARN] AutoProcessor failed ({e}), falling back to AutoTokenizer for processor")
    print("[WARN] Image preprocessing may be limited without AutoProcessor")
    processor = AutoTokenizer.from_pretrained(
        BASE_MODEL_NAME, trust_remote_code=True
    )

qwen_cfg = AutoConfig.from_pretrained(
    BASE_MODEL_NAME, trust_remote_code=True
)

text_cfg = build_text_config_from_qwen(qwen_cfg)
vision_cfg = build_vision_config_from_qwen(qwen_cfg)
infini_cfg = InfiniDopamineConfig(
    text_config=text_cfg,
    vision_config=vision_cfg,
    image_token_id=getattr(qwen_cfg, "image_token_id", 248056),
    video_token_id=getattr(qwen_cfg, "video_token_id", 248057),
    vision_start_token_id=getattr(qwen_cfg, "vision_start_token_id", 248053),
    vision_end_token_id=getattr(qwen_cfg, "vision_end_token_id", 248054),
)

model = InfiniDopamineForConditionalGeneration(infini_cfg)

print("Loading Qwen3.5-0.8B weights into InfiniDopamine...")
try:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=TORCH_DTYPE,
        device_map="cpu",
        trust_remote_code=True,
    )
except Exception as e:
    print(f"[WARN] AutoModelForCausalLM failed ({e}), trying AutoModelForVision2Seq")
    from transformers import AutoModelForVision2Seq
    base_model = AutoModelForVision2Seq.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=TORCH_DTYPE,
        device_map="cpu",
        trust_remote_code=True,
    )

# The built-in loader handles prefix remapping and GDN-1 -> GDN-2 conversion.
missing, unexpected = model.load_qwen35_weights(base_model, strict=False)

del base_model
gc.collect()
torch.cuda.empty_cache()

print(f"\nModel type       : {model.config.model_type}")
print(f"Total params     : {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Missing keys     : {len(missing):,}")
print(f"Unexpected keys  : {len(unexpected):,}")
if missing:
    print("First missing keys:", missing[:10])
if unexpected:
    print("First unexpected keys:", unexpected[:10])

## LoRA / Unsloth training preparation

We apply PEFT here and prepare the model for TRL. The trainer later receives the already-wrapped model, so `peft_config` is not passed twice.

LoRA targets include the vision tower (`merger.linear_fc1`, `merger.linear_fc2`) so the multimodal adapter can learn along with the text backbone.

In [ ]:
lora_cfg = None
if USE_LORA:
    lora_cfg = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable(use_reentrant=False)

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
else:
    def _make_inputs_require_grad(module, _in, output):
        output.requires_grad_(True)

    emb = model.get_input_embeddings() if hasattr(model, "get_input_embeddings") else None
    if emb is not None:
        emb.register_forward_hook(_make_inputs_require_grad)

if hasattr(model, "config") and model.config is not None:
    model.config.use_cache = False

model.train()
print("Model prepared for CPT.")

## Dataset formatting

Every dataset has a different schema. The `format_example()` function below handles each dataset ID explicitly, converting its native columns into a single `text` field the tokenizer can consume.

Special handling:
- **SMB**: downloads the repo, extracts `.npz` frames, serializes the 8-button action vector to text.
- **WorldModelForMaze**: clones the repo, reads tokenized maze `.json` files.
- **Sokoban**: flattens `messages[*].content` chat turns.
- **ALFWorld**: serializes `steps` JSON into an observation/action transcript.
- **Chess (laion)**: converts SAN `Moves` list into PGN-like movetext.
- **r0b0tlab**: parses `messages_json` string into chat text.

In [ ]:
# ──────────────────────────────────────────────────────────────
#  Dataset-specific formatters
# ──────────────────────────────────────────────────────────────

def _flatten_messages(messages: Union[List[Dict], str]) -> str:
    """Flatten a chat-format messages list into a single text string."""
    if isinstance(messages, str):
        try:
            messages = json.loads(messages)
        except Exception as e:
            print(f"[WARN] Dataset formatting error: {e}")
            return str(messages)
    parts = []
    for m in messages:
        role = m.get("role", m.get("from", "unknown"))
        content = m.get("content", m.get("value", ""))
        if isinstance(content, list):
            text_parts = []
            for item in content:
                if isinstance(item, dict) and item.get("type") == "text":
                    text_parts.append(item.get("text", ""))
            content = " ".join(text_parts)
        parts.append(f"{role}: {content}")
    return "\n".join(parts)


def format_smb(example: Dict) -> Dict:
    """SMB world model: serialize action vector to text."""
    action = example.get("action", [])
    if isinstance(action, np.ndarray):
        action = action.tolist()
    buttons = ["Up", "Down", "Left", "Right", "A", "B", "Start", "Select"]
    action_str = ", ".join(
        f"{b}={float(v):.1f}" for b, v in zip(buttons, action)
    ) if len(action) == len(buttons) else str(action)
    text = f"SMB Frame Action: [{action_str}]"
    return {"text": text}


def format_maze(example: Dict) -> Dict:
    """WorldModelForMaze: maze text is already a single string."""
    text = example.get("text", str(example))
    return {"text": text}


def format_sokoban(example: Dict) -> Dict:
    """Sokoban: flatten messages JSON string, include task/seed metadata."""
    messages = example.get("messages", "[]")
    task = example.get("task", "")
    seed = example.get("seed", "")
    env_id = example.get("env_id", "")
    flat = _flatten_messages(messages)
    header = "".join(
        f"{k}: {v}\n" for k, v in [("task", task), ("seed", seed), ("env_id", env_id)] if v
    )
    return {"text": header + flat}


def format_bytesized32(example: Dict) -> Dict:
    """Bytesized32: prompt list + reward_model JSON + extra_info JSON."""
    prompt = example.get("prompt", [])
    prompt_text = _flatten_messages(prompt)
    reward = example.get("reward_model", {})
    extra = example.get("extra_info", {})
    reward_text = json.dumps(reward, ensure_ascii=False) if reward else ""
    extra_text = json.dumps(extra, ensure_ascii=False) if extra else ""
    text = f"{prompt_text}\n\nReward Model: {reward_text}\nExtra: {extra_text}"
    return {"text": text}


def format_patronus(example: Dict) -> Dict:
    """PatronusAI: chat messages list."""
    text = _flatten_messages(example.get("messages", []))
    return {"text": text}


def format_arc(example: Dict) -> Dict:
    """ARC traces (TOP PRIORITY): serialize task metadata + level scores."""
    parts = []
    task = example.get("task", "")
    status = example.get("status", "")
    win_levels = example.get("win_levels", "")
    provider = example.get("provider", "")
    model = example.get("model", "")
    effort = example.get("effort", "")
    port = example.get("port", "")
    parts.append(f"ARC Task: {task}")
    if status:
        parts.append(f"Status: {status}")
    if win_levels:
        parts.append(f"Win Levels: {win_levels}")
    if provider:
        parts.append(f"Provider: {provider}")
    if model:
        parts.append(f"Model: {model}")
    if effort:
        parts.append(f"Effort: {effort}")
    if port:
        parts.append(f"Port: {port}")
    levels = []
    for lvl in range(10):
        v = example.get(f"level{lvl}")
        if v is not None:
            levels.append(f"level{lvl}={v}")
    if levels:
        parts.append("Level Scores: " + ", ".join(levels))
    return {"text": "\n".join(parts)}


def format_chess_laion(example: Dict) -> Dict:
    """Laion chess: convert SAN moves to PGN-like text."""
    moves = example.get("Moves", [])
    termination = example.get("Termination", "")
    result = example.get("Result", "*")
    if isinstance(moves, list):
        movetext = " ".join(str(m) for m in moves)
    else:
        movetext = str(moves)
    text = f"[Event \"?\"]\n[Result \"{result}\"]\n\n{movetext} {termination}"
    return {"text": text}


def format_openthoughts(example: Dict) -> Dict:
    """OpenThoughts: system + conversations chat."""
    system = example.get("system", "")
    convs = example.get("conversations", [])
    parts = []
    if system:
        parts.append(f"system: {system}")
    for turn in convs:
        role = turn.get("from", turn.get("role", "unknown"))
        value = turn.get("value", turn.get("content", ""))
        parts.append(f"{role}: {value}")
    return {"text": "\n".join(parts)}


def format_alfworld(example: Dict) -> Dict:
    """ALFWorld: parse steps JSON string, serialize trajectory."""
    steps_raw = example.get("steps", "[]")
    if isinstance(steps_raw, str):
        try:
            steps = json.loads(steps_raw)
        except Exception as e:
            print(f"[WARN] Dataset formatting error: {e}")
            steps = []
    else:
        steps = steps_raw if isinstance(steps_raw, list) else []
    parts = []
    task = example.get("task", "")
    task_type = example.get("task_type", "")
    if task:
        parts.append(f"Task: {task}")
    if task_type:
        parts.append(f"Task Type: {task_type}")
    for step in steps:
        idx = step.get("idx", step.get("step", step.get("id", "")))
        obs = step.get("obs", step.get("observation", step.get("text", "")))
        action = step.get("action", step.get("act", ""))
        if obs:
            parts.append(f"Step {idx} Observation: {obs}")
        if action:
            parts.append(f"Step {idx} Action: {action}")
    return {"text": "\n".join(parts)}


def format_kimi_k3(example: Dict) -> Dict:
    """Kimi K3 coding traces: messages chat list."""
    text = _flatten_messages(example.get("messages", []))
    return {"text": text}


def format_cot_eval(example: Dict) -> Dict:
    """CoT eval traces: passage + question + options + answer + reasoning_trace."""
    parts = []
    passage = example.get("passage", "")
    if passage:
        parts.append(f"Passage: {passage}")
    question = example.get("question", "")
    if question:
        parts.append(f"Question: {question}")
    options = example.get("options", [])
    if options:
        opts = " | ".join(str(o) for o in options)
        parts.append(f"Options: {opts}")
    answer = example.get("answer", "")
    if answer:
        parts.append(f"Answer: {answer}")
    trace = example.get("reasoning_trace", "")
    if trace:
        parts.append(f"Reasoning: {trace}")
    return {"text": "\n".join(parts)}


def format_lichess(example: Dict) -> Dict:
    """Lichess: movetext + metadata."""
    movetext = example.get("movetext", "")
    white = str(example.get("White") or "?")
    black = str(example.get("Black") or "?")
    result = example.get("Result", "*")
    opening = example.get("Opening", "")
    eco = example.get("ECO", "")
    event = example.get("Event", "")
    site = example.get("Site", "")
    date = example.get("UTCDate", "")
    text = (
        f"[Event \"{event}\"]\n[Site \"{site}\"]\n"
        f"[Date \"{date}\"]\n[White \"{white}\"]\n[Black \"{black}\"]\n"
        f"[Result \"{result}\"]\n[ECO \"{eco}\"]\n[Opening \"{opening}\"]\n\n{movetext}"
    )
    return {"text": text}


def format_toolace(example: Dict) -> Dict:
    """ToolACE: system + conversations."""
    system = example.get("system", "")
    convs = example.get("conversations", [])
    parts = []
    if system:
        parts.append(f"system: {system}")
    for turn in convs:
        role = turn.get("from", turn.get("role", "unknown"))
        value = turn.get("value", turn.get("content", ""))
        parts.append(f"{role}: {value}")
    return {"text": "\n".join(parts)}


def format_qwen3_distill(example: Dict) -> Dict:
    """Qwen3.8-27B distillation: messages chat list + metadata."""
    text = _flatten_messages(example.get("messages", []))
    domain = example.get("domain", "")
    category = example.get("category", "")
    source = example.get("source", "")
    meta = " | ".join(x for x in [f"domain={domain}", f"category={category}", f"source={source}"] if x)
    if meta:
        text = f"[{meta}]\n{text}"
    return {"text": text}


def format_fable5(example: Dict) -> Dict:
    """Fable-5: messages + trace."""
    text = _flatten_messages(example.get("messages", []))
    trace = example.get("trace", "")
    prompt = example.get("prompt", "")
    parts = []
    if prompt:
        parts.append(f"Prompt: {prompt}")
    parts.append(text)
    if trace:
        parts.append(f"Trace: {trace}")
    return {"text": "\n".join(parts)}


def format_wikitext(example: Dict) -> Dict:
    """WikiText: single text column, filter empty rows."""
    text = example.get("text", "")
    if not text or not text.strip():
        text = "[EMPTY_WIKITEXT_ROW]"
    return {"text": text}


def format_r0b0tlab(example: Dict) -> Dict:
    """r0b0tlab distillation: parse messages_json string."""
    raw = example.get("messages_json", "[]")
    if isinstance(raw, str):
        try:
            raw = json.loads(raw)
        except Exception as e:
            print(f"[WARN] Dataset formatting error: {e}")
            raw = []
    text = _flatten_messages(raw)
    # Append task_type / source metadata if present
    task_type = example.get("task_type", "")
    source = example.get("source", "")
    domain = example.get("domain", "")
    meta = " | ".join(x for x in [f"task_type={task_type}", f"source={source}", f"domain={domain}"] if x)
    if meta:
        text = f"[{meta}]\n{text}"
    return {"text": text}


# Map dataset ID -> formatter function.
DATASET_FORMATTERS = {
    "DylanRiden/smb-worldmodel-data": format_smb,
    "Kalso42/WorldModelForMaze": format_maze,
    "ultrastar111/sokoban_easy_v8_cot_chunk_kinf_world_model_20260707_perseg": format_sokoban,
    "thuml/bytesized32-world-model-cot": format_bytesized32,
    "PatronusAI/world_model_corpus": format_patronus,
    "schema-harness/arc-agi-3-schema-traces": format_arc,  # TOP PRIORITY
    "laion/strategic_game_chess": format_chess_laion,
    "ryanmarten/OpenThoughts-1k-sample": format_openthoughts,
    "Decix/ReBel-ALFWorld-SFT-Trajectories": format_alfworld,
    "greghavens/kimi-k3-coding-and-debugging-traces": format_kimi_k3,
    "cot-leaderboard/cot-eval-traces-2.0": format_cot_eval,
    "Lichess/standard-chess-games": format_lichess,
    "lockon/ToolACE": format_toolace,
    "faunix/Qwen3.8-27B-Distillation-40K": format_qwen3_distill,
    "Glint-Research/Fable-5-traces": format_fable5,
    "Salesforce/wikitext": format_wikitext,
    "r0b0tlab/qwen3.8-max-glm5.2-kimi-k3-distillation": format_r0b0tlab,
}


def format_example(example: Dict, dataset_name: str) -> Dict:
    formatter = DATASET_FORMATTERS.get(dataset_name)
    if formatter is not None:
        return formatter(example)
    # Fallback: try common text columns
    for col in ["text", "content", "prompt", "problem", "solution"]:
        if col in example and example[col]:
            return {"text": str(example[col])}
    # Last resort: stringify all scalar values
    text = " ".join(
        str(v) for v in example.values()
        if isinstance(v, (str, int, float)) and not str(v).startswith("_")
    )
    return {"text": text}


def tokenize_fn(example: Dict) -> dict:
    tok = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )
    return {
        "input_ids": tok["input_ids"],
        "attention_mask": tok["attention_mask"],
        "labels": tok["input_ids"],
    }

## Streaming dataset pipeline

Every dataset in `CPT_DATASETS` is loaded with `streaming=True` and interleaved. Special datasets (SMB, maze) are loaded outside the streaming pipeline and converted to text before interleaving.

The peek cell validates the text column; because `streaming` iterators are single-use, we rebuild the dataset afterward.

In [ ]:
# ──────────────────────────────────────────────────────────────
#  Special-case loaders for non-streaming / file-based datasets
# ──────────────────────────────────────────────────────────────

def load_smb_dataset() -> IterableDataset:
    """Download SMB repo, extract .npz frames, yield action text."""
    from huggingface_hub import hf_hub_download
    import zipfile
    
    repo_id = "DylanRiden/smb-worldmodel-data"
    cache_dir = Path(SMB_CACHE_DIR)
    cache_dir.mkdir(parents=True, exist_ok=True)
    
    zip_path = hf_hub_download(
        repo_id=repo_id,
        filename="smb_frames.zip",
        repo_type="dataset",
        cache_dir=str(cache_dir),
    )
    
    extract_dir = cache_dir / "smb_frames"
    if not extract_dir.exists():
        print(f"Extracting {zip_path} -> {extract_dir}")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)
    
    npz_files = sorted(extract_dir.rglob("*.npz"))
    print(f"Found {len(npz_files)} SMB .npz files")
    
    def _gen():
        for npz_path in npz_files:
            try:
                data = np.load(npz_path)
                action = data["action"]
                if action.ndim == 0:
                    action = np.array([0.0] * 8)
                buttons = ["Up", "Down", "Left", "Right", "A", "B", "Start", "Select"]
                action_str = ", ".join(
                    f"{b}={float(v):.1f}" for b, v in zip(buttons, action.flatten()[:8])
                )
                yield {"text": f"SMB Frame Action: [{action_str}]"}
            except Exception as e:
                print(f"[WARN] Dataset formatting error: {e}")
                continue
    
    return IterableDataset.from_generator(_gen, gen_kwargs={})


def load_maze_dataset() -> IterableDataset:
    """Clone WorldModelForMaze repo, yield maze token text."""
    from huggingface_hub import snapshot_download
    
    cache_dir = Path(MAZE_CACHE_DIR)
    cache_dir.mkdir(parents=True, exist_ok=True)
    
    maze_dir = snapshot_download(
        repo_id="Kalso42/WorldModelForMaze",
        repo_type="dataset",
        cache_dir=str(cache_dir),
    )
    maze_path = Path(maze_dir)
    # Look for .txt maze files under data/
    txt_files = sorted(maze_path.glob("data/**/*.txt"))
    if not txt_files:
        txt_files = sorted(maze_path.rglob("*.txt"))
    print(f"Found {len(txt_files)} maze .txt files")
    
    def _gen():
        for tf in txt_files:
            try:
                text = tf.read_text(encoding="utf-8", errors="replace")
                text = text.strip()
                if text:
                    yield {"text": text}
            except Exception as e:
                print(f"[WARN] Dataset formatting error: {e}")
                continue
    
    return IterableDataset.from_generator(_gen, gen_kwargs={})


DATASET_CONFIG_MAP = {
    # dataset_id -> (config, split)
    "PatronusAI/world_model_corpus": ("train", "train"),
    "Glint-Research/Fable-5-traces": ("pi_agent", "train"),
    "Salesforce/wikitext": ("wikitext-103-raw-v1", "train"),
    "r0b0tlab/qwen3.8-max-glm5.2-kimi-k3-distillation": ("sft_balanced", "train"),
}

DATASET_SUBSET_MAP = {
    # dataset_id -> max_rows (None = all)
    "Lichess/standard-chess-games": LICHESS_MAX_ROWS,
    "cot-leaderboard/cot-eval-traces-2.0": COT_EVAL_MAX_ROWS,
    "r0b0tlab/qwen3.8-max-glm5.2-kimi-k3-distillation": R0B0TLAB_MAX_ROWS,
    "Salesforce/wikitext": WIKITEXT_MAX_ROWS,
}


def _apply_subset(ds, dataset_name: str) -> IterableDataset:
    max_rows = DATASET_SUBSET_MAP.get(dataset_name)
    if max_rows is None:
        return ds
    # Take first max_rows examples from the stream
    def _gen():
        count = 0
        for ex in ds:
            yield ex
            count += 1
            if count >= max_rows:
                break
    return IterableDataset.from_generator(_gen, gen_kwargs={})


def build_streaming_dataset(
    dataset_names: List[str],
    seed: int = 42,
) -> IterableDataset:
    streams = []
    
    for name in dataset_names:
        if name == "DylanRiden/smb-worldmodel-data":
            streams.append(load_smb_dataset())
        elif name == "Kalso42/WorldModelForMaze":
            streams.append(load_maze_dataset())
        else:
            cfg, split = DATASET_CONFIG_MAP.get(name, ("default", "train"))
            ds = load_dataset(name, config=cfg, split=split, streaming=True)
            formatter = DATASET_FORMATTERS.get(name)
            if formatter is not None:
                ds = ds.map(lambda ex: format_example(ex, name), batched=False)
            ds = _apply_subset(ds, name)
            streams.append(ds)
    
    if len(streams) == 1:
        return streams[0]
    
    return interleave_datasets(
        streams,
        seed=seed,
        stopping_strategy="all_exhausted",
    )


train_dataset = build_streaming_dataset(CPT_DATASETS)
sample = next(iter(train_dataset))
print(f"Sample keys  : {list(sample.keys())}")
print(f"Sample text  : {str(sample.get('text', ''))[:240]}")
print(f"Sample length: {len(str(sample.get('text', '')))}")
del sample
gc.collect()

# Rebuild dataset for training (streaming iterators are single-use)
train_dataset = build_streaming_dataset(CPT_DATASETS)

In [ ]:
def tokenize_fn(example: Dict) -> dict:
    tok = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )
    return {
        "input_ids": tok["input_ids"],
        "attention_mask": tok["attention_mask"],
        "labels": tok["input_ids"],
    }


# Only remove columns if this is a non-streaming dataset.
cols_to_remove = None
if hasattr(train_dataset, "column_names"):
    cols_to_remove = [
        c for c in train_dataset.column_names if c not in {"text", "input_ids", "attention_mask", "labels"}
    ]

train_dataset = train_dataset.map(
    tokenize_fn,
    batched=False,
    remove_columns=cols_to_remove,
)

print(f"Tokenized columns : {train_dataset.column_names}")
print(f"Max seq length    : {MAX_SEQ_LENGTH}")

## Reward computation & custom trainer

`build_reward_values` computes per-token rewards from a **detached** base forward, then `CPTSFTTrainer` runs the main reward-conditioned forward for the CPT loss.

> **Compute cost:** this doubles forwards during CPT. Set `REWARD_EVERY_N_STEPS > 1` to reduce overhead.

In [ ]:
def build_reward_values(
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
) -> torch.Tensor:
    with torch.no_grad():
        base_outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        shift_logits = base_outputs.logits[..., :-1, :].contiguous()
        shift_labels = input_ids[..., 1:].contiguous()

        if REWARD_LOSS_TYPE == "nll":
            token_loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                reduction="none",
            ).view(input_ids.size(0), -1)
            rewards = -token_loss * REWARD_SCALE
        elif REWARD_LOSS_TYPE == "ce":
            token_loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                reduction="none",
            ).view(input_ids.size(0), -1)
            rewards = -token_loss * REWARD_SCALE
        else:
            token_loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                reduction="none",
            ).view(input_ids.size(0), -1)
            rewards = (-token_loss).exp() * REWARD_SCALE

        reward_values = torch.zeros_like(input_ids, dtype=rewards.dtype)
        reward_values[:, 1:] = rewards
        reward_values = reward_values * attention_mask
    return reward_values


class CPTSFTTrainer(SFTTrainer):
    def __init__(self, *args, reward_every_n_steps: int = 1, **kwargs):
        super().__init__(*args, **kwargs)
        self.reward_every_n_steps = max(1, int(reward_every_n_steps))
        self._global_step = 0

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs: bool = False,
        num_items_in_batch=None,
        **kwargs,
    ):
        input_ids = inputs["input_ids"].to(model.device)
        attention_mask = inputs["attention_mask"].to(model.device)
        labels = inputs.get("labels")
        if labels is not None:
            labels = labels.to(model.device)

        if self._global_step % self.reward_every_n_steps == 0:
            reward_values = build_reward_values(input_ids, attention_mask)
        else:
            reward_values = torch.zeros_like(
                input_ids, dtype=input_ids.dtype, device=model.device
            )

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            reward_values=reward_values,
        )
        self._global_step += 1
        loss = outputs.loss
        if return_outputs:
            return loss, outputs
        return loss


dummy_ids = torch.tensor(
    [tokenizer("Hello world")["input_ids"]], device="cpu"
)
dummy_mask = torch.ones_like(dummy_ids)
dummy_rewards = torch.zeros_like(dummy_ids, dtype=TORCH_DTYPE)
with torch.no_grad():
    out = model(
        input_ids=dummy_ids,
        attention_mask=dummy_mask,
        reward_values=dummy_rewards,
    )
print("Reward-values forward pass OK.")
print(f"Logits shape: {out.logits.shape}")

## Training

Launch CPT. If `RESUME_FROM_CHECKPOINT` points at a saved directory, training resumes from there.

> **Tip for Kaggle:** reduce `NUM_TRAIN_EPOCHS` to `1` and set `REWARD_EVERY_N_STEPS=2` or `4` to save compute.

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_steps=WARMUP_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    push_to_hub=PUSH_TO_HUB,
    hub_model_id=HUB_MODEL_ID or None,
    hub_token=HF_TOKEN,
    bf16=(TORCH_DTYPE == torch.bfloat16),
    fp16=(TORCH_DTYPE == torch.float16),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch_fused",
    report_to=["tensorboard"],
    seed=42,
    data_seed=42,
)

trainer = CPTSFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    reward_every_n_steps=REWARD_EVERY_N_STEPS,
)

trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)

## Save / merge / push + dataset report

- **LoRA adapters only** → smaller, standard for PEFT workflows.
- **Merged full model** → required if you want to load without PEFT later.

Both are written to `{OUTPUT_DIR}/final` by default.

The final cell prints a report of every dataset used and how it was formatted.

In [ ]:
print("\n" + "="*60)
print("DATASET REPORT — CPT Mixer")
print("="*60)

REPORT = [
    # Tier 1: must-use fully
    ("DylanRiden/smb-worldmodel-data",
     "~118k .npz frames (repo download)",
     "Downloaded repo, extracted smb_frames.zip, serialized 8-button action vectors to text."),
    ("Kalso42/WorldModelForMaze",
     "code repo + maze .txt files",
     "Cloned repo via snapshot_download, read maze text files under data/**/*.txt as text."),
    ("ultrastar111/sokoban_...",
     "train split (viewer-enabled)",
     "Flattened messages JSON string to chat text; included task/seed/env_id metadata."),
    ("thuml/bytesized32-world-model-cot",
     "301k train / 2.9k test",
     "Combined prompt list + reward_model JSON + extra_info JSON into text."),
    ("PatronusAI/world_model_corpus",
     "239k train (config=train)",
     "Flattened messages chat list (tool-use traces with system prompts). Config=train, split=train."),
    ("schema-harness/arc-agi-3-schema-traces",  # TOP PRIORITY
     "50 rows (test split)",
     "Serialized ARC benchmark metadata: task ID, status, win_levels, level0–level9 scores."),
    ("laion/strategic_game_chess",
     "train split",
     "Converted SAN Moves list + Termination + Result into PGN-like text."),
    ("ryanmarten/OpenThoughts-1k-sample",
     "1k samples",
     "Flattened system + conversations chat into text."),
    ("Decix/ReBel-ALFWorld-SFT-Trajectories",
     "426 rows",
     "Parsed steps JSON string, serialized observation/action transcript."),
    # Tier 2: important, subsetted
    ("greghavens/kimi-k3-coding-and-debugging-traces",
     "3.9k rows",
     "Flattened messages chat list (coding + debugging traces with reasoning_content)."),
    ("cot-leaderboard/cot-eval-traces-2.0",
     "3.7M rows (test split, subsetted)",
     "Combined passage + question + options + answer + reasoning_trace into text."),
    ("Lichess/standard-chess-games",
     "7.1B rows (subsetted)",
     "Converted movetext PGN + metadata (Event, White, Black, ECO, Opening) into text."),
    ("lockon/ToolACE",
     "train split",
     "Flattened system + conversations chat into text."),
    ("faunix/Qwen3.8-27B-Distillation-40K",
     "40k rows",
     "Flattened messages chat list (distilled reasoning traces) + domain/category metadata."),
    ("Glint-Research/Fable-5-traces",
     "pi_agent config, train split",
     "Flattened messages + prompt + trace into text."),
    ("Salesforce/wikitext",
     "wikitext-103-raw-v1: ~36k train (subsetted)",
     "Used single text column; empty rows replaced with [EMPTY_WIKITEXT_ROW]."),
    # Tier 3: high-value distillation
    ("r0b0tlab/qwen3.8-max-glm5.2-kimi-k3-distillation",
     "22.8M rows total (subsetted)",
     "Parsed messages_json string, flattened chat into text + task_type/source/domain metadata."),
]

for name, size, usage in REPORT:
    print(f"\nDataset : {name}")
    print(f"  Size  : {size}")
    print(f"  Usage : {usage}")

print("\n" + "="*60)
print(f"Total datasets in mixer: {len(REPORT)}")
print("="*60)